In [105]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/manaltariq13/missing-migrants-preprocessed-csv/missing_migrants_preprocessed.csv


In [106]:
df = pd.read_csv("/kaggle/input/datasets/manaltariq13/missing-migrants-preprocessed-csv/missing_migrants_preprocessed.csv")
df.head()

,Incident Type,Incident year,Reported Month,Region of Origin,Region of Incident,Country of Origin,Number of Dead,Minimum Estimated Number of Missing,Total Number of Dead and Missing,Number of Survivors,Number of Females,Number of Males,Number of Children,Cause of Death,Migration route,Location of death,Information Source,UNSD Geographical Grouping,Latitude,Longitude
0,Incident,2014,January,Central America,North America,Guatemala,1.0,0,1,0,0,1,0,Mixed or unknown,US-Mexico border crossing,Pima Country Office of the Medical Examiner ju...,Pima County Office of the Medical Examiner (PC...,Northern America,31.650259,-110.366453
1,Incident,2014,January,Latin America / Caribbean (P),North America,Unknown,1.0,0,1,0,0,0,0,Mixed or unknown,US-Mexico border crossing,Pima Country Office of the Medical Examiner ju...,Pima County Office of the Medical Examiner (PC...,Northern America,31.597130,-111.737560
2,Incident,2014,January,Latin America / Caribbean (P),North America,Unknown,1.0,0,1,0,0,0,0,Mixed or unknown,US-Mexico border crossing,Pima Country Office of the Medical Examiner ju...,Pima County Office of the Medical Examiner (PC...,Northern America,31.940260,-113.011250
3,Incident,2014,January,Central America,North America,Mexico,1.0,0,1,0,0,1,0,Violence,US-Mexico border crossing,"near Douglas, Arizona, USA","Ministry of Foreign Affairs Mexico, Pima Count...",Northern America,31.506777,-109.315632
4,Incident,2014,January,Northern Africa,Europe,Sudan,1.0,0,1,2,0,1,0,Harsh environmental conditions / lack of adequ...,Unknown,Border between Russia and Estonia,EUBusiness (Agence France-Presse),Northern Europe,59.155100,28.000000


# Fix Coordinates
Records with invalid coordinates (0,0) were removed to avoid misleading spatial clustering.

In [107]:
df = df[(df["Latitude"] != 0) & (df["Longitude"] != 0)]

# Create New Features

In [108]:
#Vulneability index
df["Vulnerability_Index"] = df["Number of Children"] + df["Number of Females"]

#Survival Ratio
df["Total_People"] = df["Total Number of Dead and Missing"] + df["Number of Survivors"]

df["Survival_Ratio"] = df["Number of Survivors"] / df["Total_People"]

df.loc[df["Total_People"] == 0, "Survival_Ratio"] = 0

#season feature
def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

df["Season"] = df["Reported Month"].apply(get_season)

#cause feature
def categorize_cause(cause):
    if pd.isna(cause):
        return "Unknown"
    cause = cause.lower()
    
    if "drown" in cause or "exposure" in cause:
        return "Environmental"
    elif "violence" in cause or "shot" in cause:
        return "Violence"
    else:
        return "Accident"

df["Cause_Category"] = df["Cause of Death"].apply(categorize_cause)

# Dropping Useless Columns

In [109]:
df.drop(columns=[
    "Location of death",
    "Information Source"
], inplace=True, errors='ignore')

# K-Means Clustering

In [110]:
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=5, random_state=42)

df["Location_Cluster"] = kmeans.fit_predict(df[["Latitude", "Longitude"]])

# Encoding All Categorical Columns

In [111]:
categorical_cols = df.select_dtypes(include=['object']).columns

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Step 1: Replace Infinity Values and Handling Nan Values
During feature engineering, division-based features (e.g., Survival Ratio) introduced infinite values when the denominator was zero. These were handled by replacing infinite values with NaN and subsequently imputing them with zero to ensure compatibility with machine learning algorithms.

In [112]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

df.fillna(0, inplace=True)

#Fill NaNs with 0 (or mean)
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)


# Feature Scaling

In [113]:

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

num_cols = df.select_dtypes(include=['int64', 'float64']).columns

df[num_cols] = scaler.fit_transform(df[num_cols])

# Train-Test Split

In [114]:
from sklearn.model_selection import train_test_split

target = "Total Number of Dead and Missing"

X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature Importance (5 Models)

In [115]:
# Import models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
import pandas as pd

# Define models
models = {
    "Random Forest": RandomForestRegressor(),
    "Decision Tree": DecisionTreeRegressor(),
    "Gradient Boosting": GradientBoostingRegressor(),
    "XGBoost": XGBRegressor(),
}

# Dictionary to store feature importances
importances = {}

# Fit each model and store feature importances
for name, model in models.items():
    model.fit(X_train, y_train)
    importances[name] = model.feature_importances_

# Create a DataFrame of feature importances
importance_df = pd.DataFrame(importances, index=X_train.columns)

# Calculate average importance across all models
importance_df["Average"] = importance_df.mean(axis=1)

# Display top 10 features by average importance
top_features = importance_df.sort_values(by="Average", ascending=False).head(10)
print(top_features)

                                                    Random Forest  \
Minimum Estimated Number of Missing                      0.795063   
Number of Dead                                           0.142144   
Total_People                                             0.027643   
Country of Origin_Algeria,Bangladesh,Côte d'Ivo...       0.015087   
Survival_Ratio                                           0.004397   
Reported Month_October                                   0.000056   
Number of Survivors                                      0.001981   
Latitude                                                 0.003823   
Vulnerability_Index                                      0.000355   
Region of Origin_Unknown                                 0.000243   

                                                    Decision Tree  \
Minimum Estimated Number of Missing                  7.740545e-01   
Number of Dead                                       1.291461e-01   
Total_People                     

In [116]:
import re
from lightgbm import LGBMRegressor

# Function to clean column names (letters, numbers, underscores)
def clean_column_name(name):
    return re.sub(r'[^0-9a-zA-Z_]', '_', name)

# Function to make column names unique
def make_unique_columns(columns):
    seen = {}
    new_columns = []
    for col in columns:
        if col not in seen:
            seen[col] = 1
            new_columns.append(col)
        else:
            count = seen[col]
            new_col = f"{col}_{count}"
            while new_col in seen:
                count += 1
                new_col = f"{col}_{count}"
            seen[col] += 1
            seen[new_col] = 1
            new_columns.append(new_col)
    return new_columns

# Clean and make unique column names
X_train_clean = X_train.copy()
X_train_clean.columns = make_unique_columns([clean_column_name(c) for c in X_train_clean.columns])

X_test_clean = X_test.copy()
X_test_clean.columns = make_unique_columns([clean_column_name(c) for c in X_test_clean.columns])

# Fit LightGBM safely
lgb_model = LGBMRegressor()
lgb_model.fit(X_train_clean, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001824 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1539
[LightGBM] [Info] Number of data points in the train set: 10386, number of used features: 124
[LightGBM] [Info] Start training from score 0.000714


LGBMRegressor()

# Save Final Dataset

In [117]:
df.to_csv("missing_migrants_feature_engineered.csv", index=False)